# ML Training Workflow

End-to-end LightGBM training using pipeline-generated features and triple-barrier labels.

1. **Setup** - Configure markets, date range, and label parameters
2. **Load Data** - Read features and labels from parquet store
3. **Explore Data** - Check distributions, missing values, correlations
4. **Preprocess** - Handle NaN, split train/test
5. **Train** - LightGBM with early stopping
6. **Evaluate** - Metrics, ROC, threshold analysis
7. **Feature Importance** - Identify top predictors

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from training import Dataset, LGBMTrainer, Evaluator
from qate.core.model import Market, ExchangeName, Symbol
from qate.env import sys_env
from qate.store.parquet_store import ParquetStore
from qate.util.dt_range import DtRange
from triple_barrier.triple_barrier import TripleBarrierConfig

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
%matplotlib inline

# Markets
FEATURE_MARKET = Market(ExchangeName.BINANCE, Symbol.XRP_USDT)
LABEL_MARKET = Market(ExchangeName.GMO, Symbol.XRP_JPY)
EXECUTION_SIZE = '150'

TIMEZONE = 'Asia/Tokyo'

START_DATE = "20251217"
END_DATE = "20251220"
DT_RANGE = DtRange.from_strings(START_DATE, END_DATE)


CONFIG = TripleBarrierConfig()
TARGET_COLUMN = 'label'

print(f'Feature market: {FEATURE_MARKET.id}')
print(f'Label market:   {LABEL_MARKET.id}')
print(f'Date range:     {DT_RANGE.start_str} -> {DT_RANGE.end_str}')
print(f'Label param_id: {CONFIG.param_id}')

## 1. Load Data

Read features and labels from parquet, normalize timezones.

In [ ]:
parquet_store = ParquetStore(sys_env.get_parquet_root_dir())

# Load features
feature_store = parquet_store.create_feature(FEATURE_MARKET)
features_df = feature_store.read(DT_RANGE)

# Load labels (param_id must match the pipeline run)
label_store = parquet_store.create_label('TripleBarrierLabel', LABEL_MARKET, CONFIG.param_id)
labels_df = label_store.read(DT_RANGE)

print(f'Features: {features_df.shape[0]:,} rows x {features_df.shape[1]} columns')
print(f'Labels:   {labels_df.shape[0]:,} rows x {labels_df.shape[1]} columns')

def prepare_df(df):
    """Ensure DatetimeIndex in UTC, handle ts column if present."""
    if 'ts' in df.columns:
        df.index = pd.to_datetime(df['ts'], unit='s')
        df = df.drop(columns=['ts'])
    return df.sort_index()

features_df = prepare_df(features_df)
labels_df = prepare_df(labels_df)

print(f'\nFeatures range: {features_df.index.min()} -> {features_df.index.max()}')
print(f'Labels range:   {labels_df.index.min()} -> {labels_df.index.max()}')

# Original label distribution
print('\nTriple-barrier label distribution:')
print(labels_df[TARGET_COLUMN].value_counts().sort_index().to_string())
print(f'  1 = Win, -1 = Loss, 0 = Timeout')

display(features_df.head())
display(labels_df.head())

### 1.1 Align Features and Labels, Filter by Vol/Spread

1. Align features with labels using `merge_asof` (nearest timestamp matching)
2. Include `volatility_ema` and `spread_pct_ema` in the merged result
3. Filter to keep only rows where `volatility_ema > spread_pct_ema * min_ratio`

This ensures each feature is matched to its true nearest label before filtering.

In [ ]:
# Create Dataset and align features with labels
# Include volatility_ema and spread_pct_ema for post-alignment filtering
dataset = Dataset(
    features_df=features_df,
    labels_df=labels_df,
    target_column=TARGET_COLUMN,
)

# Align and include filter columns
dataset.align(include_label_columns=['volatility_ema', 'spread_pct_ema'], direction='backward')

X, y = dataset.get_aligned_data()
print(f'After alignment: {X.shape[0]:,} rows x {X.shape[1]} columns')
print(f'  Label distribution: {y.value_counts().sort_index().to_dict()}')

# Preserve original triple-barrier labels for later win-rate analysis
y_tb = y.copy()

# Filter by vol/spread ratio
dataset.filter_by_vol_spread(min_ratio=CONFIG.min_vol_spread_ratio)

X, y = dataset.get_aligned_data()
print(f'\nAfter vol/spread filtering (ratio={CONFIG.min_vol_spread_ratio}):')
print(f'  {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'  Label distribution: {y.value_counts().sort_index().to_dict()}')

# Convert to multiclass format for LightGBM
# Maps: -1 (Short) -> 0, 0 (NoEntry) -> 1, 1 (Long) -> 2
dataset.convert_to_multiclass()

X, y = dataset.get_aligned_data()
print(f'\nAfter multiclass conversion:')
print(f'  {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'  Label distribution: 0=Short, 1=NoEntry, 2=Long')
print(f'  {y.value_counts().sort_index().to_dict()}')

## 2. Explore Data

In [ ]:
# Missing values
na_pct = (X.isna().sum() / len(X) * 100).sort_values(ascending=False)
na_nonzero = na_pct[na_pct > 0]

if len(na_nonzero) > 0:
    print(f'{len(na_nonzero)} columns with missing values (top 20):')
    print(na_nonzero.head(20).to_string())
else:
    print('No missing values.')

In [ ]:
# Feature statistics
X.describe().T

In [ ]:
# Target distribution over time
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

y_tb.value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Target Distribution')
axes[0].set_xlabel('Class (-1=Loss, 0=Timeout, 1=Win)')
axes[0].set_ylabel('Count')

(y_tb == 1).rolling(500).mean().plot(ax=axes[1], alpha=0.8)
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
axes[1].set_title('Win Rate Over Time (rolling 500)')
axes[1].set_ylabel('Win Rate')

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation with target (top 20)
correlations = X.corrwith(y).abs().sort_values(ascending=False)
print('Top 20 features correlated with target:')
print(correlations.head(20).to_string())

In [ ]:
# Correlation heatmap of top features
top_features = correlations.head(15).index.tolist()

plt.figure(figsize=(12, 10))
corr_matrix = X[top_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix (Top 15 by Target Correlation)')
plt.tight_layout()
plt.show()

## 3. Preprocess Data

In [ ]:
# Drop columns with >50% missing, forward-fill the rest
dataset.drop_na_features(threshold=0.5)
dataset.fill_na(method='ffill')

X, y = dataset.get_aligned_data()
print(f'After preprocessing: {X.shape[0]:,} rows, {X.shape[1]} features')
print(f'Remaining NaN: {X.isna().sum().sum()}')
# Align original triple-barrier labels to the post-preprocessing feature index
y_tb = y_tb.loc[X.index]


In [ ]:
# Time-series train/test split (no shuffle)
dataset.train_test_split(test_size=0.2, shuffle=False)

X_train, y_train = dataset.get_train_data()
X_test, y_test = dataset.get_test_data()

print(f'Train: {len(X_train):,} samples  ({X_train.index.min()} -> {X_train.index.max()})')
print(f'Test:  {len(X_test):,} samples  ({X_test.index.min()} -> {X_test.index.max()})')
print(f'\nTrain win rate: {y_train.mean():.2%}')
print(f'Test  win rate: {y_test.mean():.2%}')

## 4. Train Model

In [ ]:
# Multiclass classification: 0=Short, 1=NoEntry, 2=Long
trainer = LGBMTrainer(
    task='classification',
    params={
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'n_jobs': -1,
        'random_state': 42,
    },
    num_boost_round=1000,
    early_stopping_rounds=50,
    verbose_eval=100,
)

trainer.fit(
    X_train, y_train,
    X_val=X_test, y_val=y_test,
)

print(f'\nBest iteration: {trainer.best_iteration}')

In [ ]:
# Training history
if trainer.training_history:
    fig, ax = plt.subplots(figsize=(10, 5))
    for name, metrics in trainer.training_history.items():
        for metric, values in metrics.items():
            ax.plot(values, label=f'{name}_{metric}')
    if trainer.best_iteration:
        ax.axvline(x=trainer.best_iteration, color='r', linestyle='--',
                   label=f'Best iter ({trainer.best_iteration})')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.set_title('Training History')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Evaluate Model

In [ ]:
# Multiclass evaluation
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Get predictions (multiclass returns probabilities for each class)
y_proba = trainer.predict(X_test)  # Shape: (n_samples, 3)
y_pred = np.argmax(y_proba, axis=1)

# Class names for display
class_names = ['Short (0)', 'NoEntry (1)', 'Long (2)']

print('='*60)
print('Classification Report')
print('='*60)
print(classification_report(y_test, y_pred, target_names=class_names))

print(f'\nOverall Accuracy: {accuracy_score(y_test, y_pred):.4f}')

In [ ]:
# Confusion matrix (3x3 for multiclass)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred Short', 'Pred NoEntry', 'Pred Long'],
    yticklabels=['Actual Short', 'Actual NoEntry', 'Actual Long'],
)
plt.title('Confusion Matrix (3-class)')
plt.tight_layout()
plt.show()

# Show class-wise accuracy
print('\nClass-wise accuracy:')
for i, name in enumerate(class_names):
    class_mask = y_test == i
    if class_mask.sum() > 0:
        class_acc = (y_pred[class_mask] == i).mean()
        print(f'  {name}: {class_acc:.2%} ({class_mask.sum():,} samples)')

In [ ]:
# ROC curves for multiclass (One-vs-Rest)
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Binarize labels for OvR ROC
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot ROC for each class
colors = ['red', 'orange', 'green']
for i, (name, color) in enumerate(zip(class_names, colors)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, label=f'{name} (AUC={roc_auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves (One-vs-Rest)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Class probability distributions
for i, (name, color) in enumerate(zip(class_names, colors)):
    axes[1].hist(y_proba[:, i], bins=50, alpha=0.5, color=color, label=name)
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Count')
axes[1].set_title('Probability Distribution by Class')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Confidence analysis for multiclass
# Confidence = max probability across classes

confidence = np.max(y_proba, axis=1)
correct = (y_pred == y_test.values)

# Create analysis DataFrame
analysis_df = pd.DataFrame({
    'y_true': y_test.values,
    'y_pred': y_pred,
    'confidence': confidence,
    'correct': correct,
    'prob_short': y_proba[:, 0],
    'prob_noentry': y_proba[:, 1],
    'prob_long': y_proba[:, 2],
})

# Accuracy by confidence level
print('Accuracy by confidence level:')
analysis_df['confidence_bin'] = pd.cut(analysis_df['confidence'], bins=[0, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0])
conf_acc = analysis_df.groupby('confidence_bin', observed=True)['correct'].agg(['mean', 'count'])
print(conf_acc.to_string())

print(f'\nMean confidence: {confidence.mean():.3f}')
print(f'High confidence (>0.6) samples: {(confidence > 0.6).sum():,} ({(confidence > 0.6).mean():.1%})')

In [ ]:
# Confidence vs accuracy plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy vs confidence threshold
thresholds = np.arange(0.35, 0.95, 0.05)
accs, coverages = [], []
for thresh in thresholds:
    mask = confidence >= thresh
    if mask.sum() > 0:
        accs.append(correct[mask].mean())
        coverages.append(mask.mean())
    else:
        accs.append(np.nan)
        coverages.append(0)

ax1 = axes[0]
ax2 = ax1.twinx()
ax1.plot(thresholds, accs, 'b-', marker='o', label='Accuracy')
ax2.plot(thresholds, coverages, 'r--', marker='s', label='Coverage')
ax1.set_xlabel('Confidence Threshold')
ax1.set_ylabel('Accuracy', color='b')
ax2.set_ylabel('Coverage (fraction of samples)', color='r')
ax1.set_title('Accuracy vs Coverage Trade-off')
ax1.legend(loc='lower left')
ax2.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Prediction distribution by actual class
pred_by_true = pd.crosstab(analysis_df['y_true'], analysis_df['y_pred'], normalize='index')
pred_by_true.plot(kind='bar', ax=axes[1], color=['red', 'orange', 'green'])
axes[1].set_xlabel('Actual Class')
axes[1].set_ylabel('Fraction')
axes[1].set_title('Prediction Distribution by Actual Class')
axes[1].set_xticklabels(['Short', 'NoEntry', 'Long'], rotation=0)
axes[1].legend(['Pred Short', 'Pred NoEntry', 'Pred Long'])

plt.tight_layout()
plt.show()

### 5.1 Trading Simulation

Simulate trading using the model predictions:
- **Predicted Long (2)**: Enter long position, exit at PT/SL or timeout
- **Predicted Short (0)**: Enter short position, exit at PT/SL or timeout  
- **Predicted NoEntry (1)**: Skip, no trade

PnL calculation:
- Correct direction prediction: Use actual PnL from triple-barrier labels
- Wrong direction prediction: Inverse of actual PnL (we took opposite position)
- Trade on NoEntry: Lose approximately half the spread

In [ ]:
# Trading simulation using model predictions
# For each test sample, simulate trading based on prediction

# Get original labels (before multiclass conversion) for PnL lookup
# Labels in labels_df: 1=Long, -1=Short, 0=NoEntry
# Need to match test indices to labels_df

# Get test timestamps
test_indices = X_test.index

# Lookup original labels and pnl from labels_df using merge_asof
test_lookup = pd.DataFrame({'time': test_indices}).reset_index(drop=True)
test_lookup['time'] = pd.to_datetime(test_lookup['time'])

labels_lookup = labels_df[['label', 'pnl', 'volatility_ema', 'spread_pct_ema']].reset_index()
labels_lookup.columns = ['time', 'original_label', 'pnl', 'volatility_ema', 'spread_pct_ema']

# Merge to get original labels and pnl for each test sample
merged = pd.merge_asof(
    test_lookup,
    labels_lookup,
    on='time',
    tolerance=pd.Timedelta('1s'),
    direction='nearest'
)

# Add predictions
merged['y_pred'] = y_pred  # 0=Short, 1=NoEntry, 2=Long
merged['y_true_multiclass'] = y_test.values  # Already converted: 0=Short, 1=NoEntry, 2=Long

# Calculate simulated PnL based on prediction vs actual
def calculate_sim_pnl(row):
    """
    Calculate PnL for a simulated trade based on prediction.
    
    Prediction mapping: 0=Short, 1=NoEntry, 2=Long
    Original label: -1=Short, 0=NoEntry, 1=Long
    
    PnL in labels_df:
    - Label 1 (Long win): PnL = bid(exit) - ask(entry) > 0
    - Label -1 (Short win): PnL = bid(entry) - ask(exit) > 0
    - Label 0 (Timeout): PnL = 0
    """
    pred = row['y_pred']
    original_label = row['original_label']
    original_pnl = row['pnl'] if not pd.isna(row['pnl']) else 0
    spread_cost = row['spread_pct_ema'] * 0.5 if not pd.isna(row['spread_pct_ema']) else 0.0001
    
    # NoEntry prediction - no trade, no pnl
    if pred == 1:
        return 0.0
    
    # Predicted Long (pred=2)
    if pred == 2:
        if original_label == 1:  # Actual was Long win - correct prediction
            return original_pnl
        elif original_label == -1:  # Actual was Short win - wrong direction
            return -abs(original_pnl)  # We lose what the short would have gained
        else:  # original_label == 0 (Timeout/NoEntry)
            return -spread_cost  # Trade during low vol, lose spread
    
    # Predicted Short (pred=0)
    if pred == 0:
        if original_label == -1:  # Actual was Short win - correct prediction
            return original_pnl
        elif original_label == 1:  # Actual was Long win - wrong direction
            return -abs(original_pnl)  # We lose what the long would have gained
        else:  # original_label == 0 (Timeout/NoEntry)
            return -spread_cost  # Trade during low vol, lose spread
    
    return 0.0

merged['sim_pnl'] = merged.apply(calculate_sim_pnl, axis=1)
merged['cumulative_pnl'] = merged['sim_pnl'].cumsum()

# Also calculate baseline: always trade Long
merged['baseline_pnl'] = merged.apply(
    lambda row: row['pnl'] if row['original_label'] == 1 else 
                (-abs(row['pnl']) if row['original_label'] == -1 else 
                 -row['spread_pct_ema'] * 0.5 if not pd.isna(row['spread_pct_ema']) else 0),
    axis=1
)
merged['cumulative_baseline'] = merged['baseline_pnl'].cumsum()

# Random strategy baseline (trade Long/Short randomly based on class distribution)
np.random.seed(42)
random_preds = np.random.choice([0, 1, 2], size=len(merged), p=[1/3, 1/3, 1/3])
merged['y_pred_random'] = random_preds
merged['random_pnl'] = merged.apply(
    lambda row: calculate_sim_pnl(pd.Series({
        'y_pred': row['y_pred_random'],
        'original_label': row['original_label'],
        'pnl': row['pnl'],
        'spread_pct_ema': row['spread_pct_ema']
    })), axis=1
)
merged['cumulative_random'] = merged['random_pnl'].cumsum()

# Summary statistics
trade_mask = merged['y_pred'] != 1  # Trades taken (not NoEntry)
print('='*60)
print('Trading Simulation Summary')
print('='*60)
print(f"\nTotal test samples: {len(merged):,}")
print(f"Trades taken (pred != NoEntry): {trade_mask.sum():,} ({trade_mask.mean():.1%})")
print(f"Trades skipped (pred = NoEntry): {(~trade_mask).sum():,} ({(~trade_mask).mean():.1%})")

print(f"\nModel Strategy:")
print(f"  Total PnL: {merged['sim_pnl'].sum():.6f}")
print(f"  Avg PnL per trade: {merged.loc[trade_mask, 'sim_pnl'].mean():.8f}")
print(f"  Win rate: {(merged.loc[trade_mask, 'sim_pnl'] > 0).mean():.2%}")

print(f"\nAlways-Long Baseline:")
print(f"  Total PnL: {merged['baseline_pnl'].sum():.6f}")
print(f"  Avg PnL per trade: {merged['baseline_pnl'].mean():.8f}")

print(f"\nRandom Strategy Baseline:")
print(f"  Total PnL: {merged['random_pnl'].sum():.6f}")

In [ ]:
# Cumulative PnL Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot 1: Cumulative PnL comparison
ax1 = axes[0]
ax1.plot(merged['time'], merged['cumulative_pnl'], 'b-', label='Model Strategy', linewidth=1.5)
ax1.plot(merged['time'], merged['cumulative_baseline'], 'gray', label='Always Long', alpha=0.5, linewidth=1)
ax1.plot(merged['time'], merged['cumulative_random'], 'orange', label='Random', alpha=0.5, linewidth=1)
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax1.set_ylabel('Cumulative PnL')
ax1.set_title('Trading Simulation: Cumulative PnL Over Time')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Plot 2: Rolling win rate and trade frequency
ax2 = axes[1]
window = 500

# Rolling win rate
trade_pnl = merged['sim_pnl'].copy()
trade_pnl[merged['y_pred'] == 1] = np.nan  # Mask NoEntry predictions
rolling_winrate = (trade_pnl > 0).rolling(window, min_periods=100).mean()
ax2.plot(merged['time'], rolling_winrate, 'g-', label='Win Rate (rolling)', linewidth=1)
ax2.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='50% baseline')

# Trade frequency
trade_taken = (merged['y_pred'] != 1).astype(float)
rolling_trade_freq = trade_taken.rolling(window, min_periods=100).mean()
ax2_twin = ax2.twinx()
ax2_twin.plot(merged['time'], rolling_trade_freq, 'purple', label='Trade Frequency', alpha=0.5)
ax2_twin.set_ylabel('Trade Frequency', color='purple')

ax2.set_xlabel('Time')
ax2.set_ylabel('Win Rate', color='green')
ax2.set_title(f'Rolling Win Rate & Trade Frequency (window={window})')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Distribution of PnL per trade
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PnL distribution
trade_pnl_values = merged.loc[merged['y_pred'] != 1, 'sim_pnl']
axes[0].hist(trade_pnl_values, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='r', linestyle='--', alpha=0.5)
axes[0].axvline(x=trade_pnl_values.mean(), color='g', linestyle='-', label=f'Mean: {trade_pnl_values.mean():.6f}')
axes[0].set_xlabel('PnL per Trade')
axes[0].set_ylabel('Count')
axes[0].set_title('PnL Distribution (Trades Only)')
axes[0].legend()

# PnL by prediction type
pred_pnl = merged.groupby('y_pred')['sim_pnl'].agg(['sum', 'mean', 'count'])
pred_pnl.index = ['Short', 'NoEntry', 'Long']
pred_pnl['sum'].plot(kind='bar', ax=axes[1], color=['red', 'gray', 'green'], edgecolor='black')
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1].set_xlabel('Prediction Type')
axes[1].set_ylabel('Total PnL')
axes[1].set_title('Total PnL by Prediction Type')
axes[1].set_xticklabels(['Short', 'NoEntry', 'Long'], rotation=0)

plt.tight_layout()
plt.show()

# Print breakdown
print('\nPnL Breakdown by Prediction Type:')
print(pred_pnl.to_string())

## 6. Feature Importance

In [ ]:
importance_df = trainer.feature_importance(importance_type='gain')
print('Top 20 important features:')
importance_df.head(20)

In [ ]:
top_n = min(20, len(importance_df))
top_imp = importance_df.head(top_n)

plt.figure(figsize=(10, 8))
plt.barh(range(top_n), top_imp['importance'].values)
plt.yticks(range(top_n), top_imp['feature'].values)
plt.xlabel('Importance (Gain)')
plt.title(f'Top {top_n} Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Prediction Analysis

In [ ]:
# Prediction analysis summary
print('Prediction Summary:')
print('='*50)

# Prediction counts
print('\nPrediction distribution:')
for i, name in enumerate(class_names):
    count = (y_pred == i).sum()
    pct = count / len(y_pred) * 100
    print(f'  {name}: {count:,} ({pct:.1f}%)')

# When model predicts Long, what's the actual distribution?
print('\nWhen model predicts Long (class 2):')
long_mask = y_pred == 2
if long_mask.sum() > 0:
    actual_dist = y_test[long_mask].value_counts().sort_index()
    for cls, count in actual_dist.items():
        print(f'  Actual {class_names[cls]}: {count:,} ({count/long_mask.sum()*100:.1f}%)')

# When model predicts Short, what's the actual distribution?
print('\nWhen model predicts Short (class 0):')
short_mask = y_pred == 0
if short_mask.sum() > 0:
    actual_dist = y_test[short_mask].value_counts().sort_index()
    for cls, count in actual_dist.items():
        print(f'  Actual {class_names[cls]}: {count:,} ({count/short_mask.sum()*100:.1f}%)')

In [ ]:
# Probability distributions by actual class
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (name, color) in enumerate(zip(class_names, colors)):
    ax = axes[i]
    for j, (actual_name, actual_color) in enumerate(zip(class_names, colors)):
        mask = y_test == j
        ax.hist(y_proba[mask, i], bins=30, alpha=0.5, color=actual_color, 
                label=f'Actual {actual_name}', density=True)
    ax.set_xlabel(f'P({name})')
    ax.set_ylabel('Density')
    ax.set_title(f'P({name}) by Actual Class')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Predictions over time
analysis_ts = analysis_df.copy()
analysis_ts.index = X_test.index

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
window = 500

# Rolling accuracy
rolling_acc = analysis_ts['correct'].rolling(window).mean()
axes[0].plot(analysis_ts.index, rolling_acc, color='blue')
axes[0].axhline(y=1/3, color='r', linestyle='--', alpha=0.5, label='Random (33%)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title(f'Rolling Accuracy (window={window})')
axes[0].legend()
axes[0].set_ylim(0, 0.6)

# Rolling class predictions
for i, (name, color) in enumerate(zip(class_names, colors)):
    pred_rate = (analysis_ts['y_pred'] == i).rolling(window).mean()
    axes[1].plot(analysis_ts.index, pred_rate, color=color, label=name, alpha=0.7)
axes[1].set_ylabel('Prediction Rate')
axes[1].set_title(f'Rolling Prediction Distribution (window={window})')
axes[1].legend()

# Rolling confidence
rolling_conf = analysis_ts['confidence'].rolling(window).mean()
axes[2].plot(analysis_ts.index, rolling_conf, color='purple')
axes[2].axhline(y=1/3, color='r', linestyle='--', alpha=0.5, label='Random baseline')
axes[2].set_ylabel('Confidence')
axes[2].set_title(f'Rolling Mean Confidence (window={window})')
axes[2].set_xlabel('Time')

plt.tight_layout()
plt.show()

## 8. Cross-Validation (Optional)

In [ ]:
# Cross-validation for multiclass
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score

X_all, y_all = dataset.get_aligned_data()

tscv = TimeSeriesSplit(n_splits=5)
cv_results = {'accuracy': [], 'f1_macro': [], 'f1_weighted': []}

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_all)):
    X_cv_train, X_cv_val = X_all.iloc[train_idx], X_all.iloc[val_idx]
    y_cv_train, y_cv_val = y_all.iloc[train_idx], y_all.iloc[val_idx]
    
    # Train a fresh model for this fold
    cv_trainer = LGBMTrainer(
        task='classification',
        params={
            'objective': 'multiclass',
            'num_class': 3,
            'metric': 'multi_logloss',
            'boosting_type': 'gbdt',
            'num_leaves': 31,
            'learning_rate': 0.05,
            'verbose': -1,
            'n_jobs': -1,
        },
        num_boost_round=200,
        early_stopping_rounds=30,
        verbose_eval=0,
    )
    cv_trainer.fit(X_cv_train, y_cv_train, X_val=X_cv_val, y_val=y_cv_val)
    
    # Predict
    y_proba_cv = cv_trainer.predict(X_cv_val)
    y_pred_cv = np.argmax(y_proba_cv, axis=1)
    
    # Metrics
    cv_results['accuracy'].append(accuracy_score(y_cv_val, y_pred_cv))
    cv_results['f1_macro'].append(f1_score(y_cv_val, y_pred_cv, average='macro'))
    cv_results['f1_weighted'].append(f1_score(y_cv_val, y_pred_cv, average='weighted'))
    
    print(f'Fold {fold+1}: Acc={cv_results["accuracy"][-1]:.4f}, F1_macro={cv_results["f1_macro"][-1]:.4f}')

cv_summary = pd.DataFrame(cv_results)
print('\nCross-Validation Summary:')
print(cv_summary.describe().T[['mean', 'std']].to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cv_summary.plot(kind='box', ax=ax)
ax.axhline(y=1/3, color='r', linestyle='--', alpha=0.5, label='Random baseline')
ax.set_ylabel('Score')
ax.set_title('Cross-Validation Results (Multiclass)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Save Model

In [ ]:
MODEL_PATH = '../models/lgbm_model'
trainer.save(MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

# Verify load
loaded_trainer = LGBMTrainer.load(MODEL_PATH)
original_preds = trainer.predict(X_test)
loaded_preds = loaded_trainer.predict(X_test)

match = np.allclose(original_preds, loaded_preds)
print(f'Predictions match after reload: {match}')